# Model Mutations

A core tenant of `json2vec` is the ability to dynamically and expressively modify model architecture at any point in time.

Model developers may update, delete, add, or reset fields before or after any loop (training, validation, testing, or inference).

In [1]:
from rich.pretty import pprint

import json2vec as j2v

Let's quickly create a basic model. It includes a flower species, and history metrics of flower metrics and the flower age during the time of measurement.

In [2]:
model = j2v.Model.from_schema(
    j2v.Category("species", target=True, max_vocab_size=10),
    j2v.Array(  # include an list of flower metrics over time
        j2v.Number("age"),
        j2v.Number("sepal_length"),
        j2v.Number("petal_length"),
        max_length=10,
        name="samples",
    ),
    d_model=16,
    n_layers=1,
    n_heads=4,
    embed=True,
)

2026-05-28 16:18:33.622 | INFO     | json2vec.architecture.root:__init__:267 - initialized Model module


Before mutating any component of a model, we need to understand predicates.

All model mutations are defined with predicates. Predicates are composable and can be combined with boolean logic (`a & b`, `~b`, `a | b`) to enable complex querying and mutations, even with dozens or hundreds of fields. Predicates can be defined not only on built in attributes per request, but also based on the parents or children of nodes.

Here, the example selects all numbers, all categories except the target, the target itself, and finally the root record node.

In [3]:
print("find all fields of type `number`")
pprint(model.select(j2v.where("type") == "number"))

print("find all fields of type `category`")
pprint(model.select(j2v.where("type") == "category"))

print("find all fields of name 'species'`")
pprint(model.select(j2v.where("name") == "species"))

print("find all fields where `target=True`")
pprint(model.select(j2v.where("target")))

find all fields of type `number`


[
│   Request(
│   │   name='age',
│   │   type='number',
│   │   description=None,
│   │   embed=False,
│   │   n_heads=4,
│   │   dropout=None,
│   │   p_mask=None,
│   │   p_prune=None,
│   │   active=True,
│   │   query='[*].samples[*].age',
│   │   pooling='query',
│   │   weight=1.0,
│   │   n_linear=1,
│   │   jitter=0.0,
│   │   n_bands=8,
│   │   offset=4,
│   │   alpha=None,
│   │   objective=<Objective.mae: 'mae'>
│   ),
│   Request(
│   │   name='sepal_length',
│   │   type='number',
│   │   description=None,
│   │   embed=False,
│   │   n_heads=4,
│   │   dropout=None,
│   │   p_mask=None,
│   │   p_prune=None,
│   │   active=True,
│   │   query='[*].samples[*].sepal_length',
│   │   pooling='query',
│   │   weight=1.0,
│   │   n_linear=1,
│   │   jitter=0.0,
│   │   n_bands=8,
│   │   offset=4,
│   │   alpha=None,
│   │   objective=<Objective.mae: 'mae'>
│   ),
│   Request(
│   │   name='petal_length',
│   │   type='number',
│   │   description=None,
│   │   embed=False,
│   │   n_heads=4,
│   │   dropout=None,
│   │   p_mask=None,
│   │   p_prune=None,
│   │   active=True,
│   │   query='[*].samples[*].petal_length',
│   │   pooling='query',
│   │   weight=1.0,
│   │   n_linear=1,
│   │   jitter=0.0,
│   │   n_bands=8,
│   │   offset=4,
│   │   alpha=None,
│   │   objective=<Objective.mae: 'mae'>
│   )
]

find all fields of type `category`


[
│   Request(
│   │   name='species',
│   │   type='category',
│   │   description=None,
│   │   embed=False,
│   │   n_heads=4,
│   │   dropout=None,
│   │   p_mask=None,
│   │   p_prune=1.0,
│   │   active=True,
│   │   query='[*].species',
│   │   pooling='query',
│   │   weight=1.0,
│   │   n_linear=1,
│   │   max_vocab_size=10,
│   │   n_bands=8,
│   │   p_unavailable=0.01,
│   │   topk=[]
│   )
]

find all fields of name 'species'`


[
│   Request(
│   │   name='species',
│   │   type='category',
│   │   description=None,
│   │   embed=False,
│   │   n_heads=4,
│   │   dropout=None,
│   │   p_mask=None,
│   │   p_prune=1.0,
│   │   active=True,
│   │   query='[*].species',
│   │   pooling='query',
│   │   weight=1.0,
│   │   n_linear=1,
│   │   max_vocab_size=10,
│   │   n_bands=8,
│   │   p_unavailable=0.01,
│   │   topk=[]
│   )
]

find all fields where `target=True`


[
│   Request(
│   │   name='species',
│   │   type='category',
│   │   description=None,
│   │   embed=False,
│   │   n_heads=4,
│   │   dropout=None,
│   │   p_mask=None,
│   │   p_prune=1.0,
│   │   active=True,
│   │   query='[*].species',
│   │   pooling='query',
│   │   weight=1.0,
│   │   n_linear=1,
│   │   max_vocab_size=10,
│   │   n_bands=8,
│   │   p_unavailable=0.01,
│   │   topk=[]
│   )
]


## Updates
`model.update(*predicates, **updates)` applies explicit schema mutations to one or more nodes. Nodes can be queried with complex predicates.

Common update APIs and field changes:

| Mutation | Purpose |
| --- | --- |
| `target=True` | Withhold a field and decode it as a supervised target. |
| `p_mask=...` | Randomly hide values for self-supervised reconstruction. |
| `p_prune=...` | Remove whole field instances from input. |
| `embed=True` | Return embeddings from a selected node. |
| `target=False` | Clear target behavior so the field is encoded as an input when present. |
| `active=False` | Reversibly remove a leaf field from encoding, forward passes, losses, and predictions. |

Users may make such updates temporary by using a context manager `with model.override(...)`.

In [4]:
model.update(j2v.where("type") == "number", p_mask=0.15)

model.update((j2v.where("type") == "category") & j2v.where("target"), p_mask=0.05)

model.update(j2v.where("target"), target=False)

model.plot()

2026-05-28 16:18:33.710 | INFO     | json2vec.architecture.mutations:_log_attribute_changes:288 - mutated schema node attribute
2026-05-28 16:18:33.710 | INFO     | json2vec.architecture.mutations:_log_attribute_changes:288 - mutated schema node attribute
2026-05-28 16:18:33.711 | INFO     | json2vec.architecture.mutations:_log_attribute_changes:288 - mutated schema node attribute
2026-05-28 16:18:33.781 | INFO     | json2vec.architecture.mutations:_log_attribute_changes:288 - mutated schema node attribute
2026-05-28 16:18:33.850 | INFO     | json2vec.architecture.mutations:_log_attribute_changes:288 - mutated schema node attribute


JSON2Vec Schema
record [array] embed
record
d_model=16  attention=mha  max_length=1  n_outputs=1  n_layers=1  n_heads=4  batch_size=1  parameters=29,334  arrays=2  
fields=4  targets=0  embeds=1  embed=True  n_linear=1
├── species [category] 3,857 params
│   record/species
│   query=[*].species  pooling=query  max_vocab_size=10  topk=[]  weight=1.0  n_heads=4  p_mask=0.05  n_linear=1  
│   n_bands=8  p_unavailable=0.01
└── samples [array] 6,608 params
    record/samples
    attention=mha  max_length=10  n_outputs=1  n_layers=1  n_heads=4  n_linear=1
    ├── age [number] 4,087 params
    │   record/samples/age
    │   query=[*].samples[*].age  pooling=query  objective=mae  weight=1.0  n_heads=4  p_mask=0.15  n_linear=1  
    │   jitter=0.0  n_bands=8  offset=4
    ├── sepal_length [number] 4,087 params
    │   record/samples/sepal_length
    │   query=[*].samples[*].sepal_length  pooling=query  objective=mae  weight=1.0  n_heads=4  p_mask=0.15  n_linear=1 
    │   jitter=0.0  n_bands=8  offset=4
    └── petal_length [number] 4,087 params
        record/samples/petal_length
        query=[*].samples[*].petal_length  pooling=query  objective=mae  weight=1.0  n_heads=4  p_mask=0.15  n_linear=1 
        jitter=0.0  n_bands=8  offset=4

`model.override(*predicates, **updates)` is for temporary mutations. The schema and runtime modules are restored when the context exits.

This is useful for reversible updates, such as temporarily deactivating a feature to determine its relative value-add to a model.

With `json2vec`, you do not need to train multiple models for optimal feature selection. You may `delete` or "deactivate" them dynamically with a one-liner.

In [5]:
with model.override(j2v.where("address") == "record/species", active=False):
    print("with overrides")
    pprint(model.select(j2v.where("address") == "record/species"))

print("without overrides")
pprint(model.select(j2v.where("address") == "record/species"))

2026-05-28 16:18:33.928 | INFO     | json2vec.architecture.mutations:_log_attribute_changes:288 - mutated schema node attribute


with overrides


[
│   Request(
│   │   name='species',
│   │   type='category',
│   │   description=None,
│   │   embed=False,
│   │   n_heads=4,
│   │   dropout=None,
│   │   p_mask=0.05,
│   │   p_prune=None,
│   │   active=False,
│   │   query='[*].species',
│   │   pooling='query',
│   │   weight=1.0,
│   │   n_linear=1,
│   │   max_vocab_size=10,
│   │   n_bands=8,
│   │   p_unavailable=0.01,
│   │   topk=[]
│   )
]

2026-05-28 16:18:34.000 | INFO     | json2vec.architecture.mutations:_log_attribute_changes:288 - restored schema node attribute


without overrides


[
│   Request(
│   │   name='species',
│   │   type='category',
│   │   description=None,
│   │   embed=False,
│   │   n_heads=4,
│   │   dropout=None,
│   │   p_mask=0.05,
│   │   p_prune=None,
│   │   active=True,
│   │   query='[*].species',
│   │   pooling='query',
│   │   weight=1.0,
│   │   n_linear=1,
│   │   max_vocab_size=10,
│   │   n_bands=8,
│   │   p_unavailable=0.01,
│   │   topk=[]
│   )
]

## Extending Model Tree

`json2vec` is not limited to merely updating the model architecture. Users may also extend the tree by adding nodes to it.

You may add a node to the tree by its target destination and the new node definition.

Use `model.extend(*predicates, *requests)` to add fields after the model already exists.

This is a premiere feature of `json2vec` because it enables continuous schema evolution with a one-liner. If new features become available, you may easily mutate the model's architecture to integrate them.

Node extension is uniquely powerful in the case of maintaining an organizational foundation model (customer behavior at a bank) while also being able to add use-case specific data targets as a new node in the model tree.

In [6]:
# add a new number "sepal_width" to the array of samples
model.extend(j2v.where("name") == "samples", j2v.Number("sepal_width", p_mask=0.15))

model.plot()

2026-05-28 16:18:34.079 | INFO     | json2vec.architecture.mutations:_log_node_mutation:301 - extended schema node


JSON2Vec Schema
record [array] embed
record
d_model=16  attention=mha  max_length=1  n_outputs=1  n_layers=1  n_heads=4  batch_size=1  parameters=33,421  arrays=2  
fields=5  targets=0  embeds=1  embed=True  n_linear=1
├── species [category] 3,857 params
│   record/species
│   query=[*].species  pooling=query  max_vocab_size=10  topk=[]  weight=1.0  n_heads=4  p_mask=0.05  n_linear=1  
│   n_bands=8  p_unavailable=0.01
└── samples [array] 6,608 params
    record/samples
    attention=mha  max_length=10  n_outputs=1  n_layers=1  n_heads=4  n_linear=1
    ├── age [number] 4,087 params
    │   record/samples/age
    │   query=[*].samples[*].age  pooling=query  objective=mae  weight=1.0  n_heads=4  p_mask=0.15  n_linear=1  
    │   jitter=0.0  n_bands=8  offset=4
    ├── sepal_length [number] 4,087 params
    │   record/samples/sepal_length
    │   query=[*].samples[*].sepal_length  pooling=query  objective=mae  weight=1.0  n_heads=4  p_mask=0.15  n_linear=1 
    │   jitter=0.0  n_bands=8  offset=4
    ├── petal_length [number] 4,087 params
    │   record/samples/petal_length
    │   query=[*].samples[*].petal_length  pooling=query  objective=mae  weight=1.0  n_heads=4  p_mask=0.15  n_linear=1 
    │   jitter=0.0  n_bands=8  offset=4
    └── sepal_width [number] 4,087 params
        record/samples/sepal_width
        query=[*].samples[*].sepal_width  pooling=query  objective=mae  weight=1.0  n_heads=4  p_mask=0.15  n_linear=1  
        jitter=0.0  n_bands=8  offset=4

In [7]:
# add a new categorical variable "caretaker" to the root of the treee
model.extend(j2v.where("name") == "record", j2v.Category("caretaker", max_vocab_size=10))

model.plot()

2026-05-28 16:18:34.205 | INFO     | json2vec.architecture.mutations:_log_node_mutation:301 - extended schema node


JSON2Vec Schema
record [array] embed
record
d_model=16  attention=mha  max_length=1  n_outputs=1  n_layers=1  n_heads=4  batch_size=1  parameters=37,278  arrays=2  
fields=6  targets=0  embeds=1  embed=True  n_linear=1
├── species [category] 3,857 params
│   record/species
│   query=[*].species  pooling=query  max_vocab_size=10  topk=[]  weight=1.0  n_heads=4  p_mask=0.05  n_linear=1  
│   n_bands=8  p_unavailable=0.01
├── samples [array] 6,608 params
│   record/samples
│   attention=mha  max_length=10  n_outputs=1  n_layers=1  n_heads=4  n_linear=1
│   ├── age [number] 4,087 params
│   │   record/samples/age
│   │   query=[*].samples[*].age  pooling=query  objective=mae  weight=1.0  n_heads=4  p_mask=0.15  n_linear=1  
│   │   jitter=0.0  n_bands=8  offset=4
│   ├── sepal_length [number] 4,087 params
│   │   record/samples/sepal_length
│   │   query=[*].samples[*].sepal_length  pooling=query  objective=mae  weight=1.0  n_heads=4  p_mask=0.15  n_linear=1 
│   │   jitter=0.0  n_bands=8  offset=4
│   ├── petal_length [number] 4,087 params
│   │   record/samples/petal_length
│   │   query=[*].samples[*].petal_length  pooling=query  objective=mae  weight=1.0  n_heads=4  p_mask=0.15  n_linear=1 
│   │   jitter=0.0  n_bands=8  offset=4
│   └── sepal_width [number] 4,087 params
│       record/samples/sepal_width
│       query=[*].samples[*].sepal_width  pooling=query  objective=mae  weight=1.0  n_heads=4  p_mask=0.15  n_linear=1  
│       jitter=0.0  n_bands=8  offset=4
└── caretaker [category] 3,857 params
    record/caretaker
    query=[*].caretaker  pooling=query  max_vocab_size=10  topk=[]  weight=1.0  n_heads=4  n_linear=1  n_bands=8  
    p_unavailable=0.01

## Deleting From Node Tree

Use `model.delete(*predicates)` when a node should leave the schema permanently. This is different from `active=False`, which keeps the node around for later reactivation.

`model.delete` is an irreversible operation. However, in some cases, it may be preferable to `active=False` because it removes parameters from the model, which will save memory and storage costs.

In [8]:
model.delete(j2v.where("name") == "petal_length")

model.plot()

2026-05-28 16:18:34.336 | INFO     | json2vec.architecture.mutations:_log_node_mutation:301 - deleted schema node


JSON2Vec Schema
record [array] embed
record
d_model=16  attention=mha  max_length=1  n_outputs=1  n_layers=1  n_heads=4  batch_size=1  parameters=33,191  arrays=2  
fields=5  targets=0  embeds=1  embed=True  n_linear=1
├── species [category] 3,857 params
│   record/species
│   query=[*].species  pooling=query  max_vocab_size=10  topk=[]  weight=1.0  n_heads=4  p_mask=0.05  n_linear=1  
│   n_bands=8  p_unavailable=0.01
├── samples [array] 6,608 params
│   record/samples
│   attention=mha  max_length=10  n_outputs=1  n_layers=1  n_heads=4  n_linear=1
│   ├── age [number] 4,087 params
│   │   record/samples/age
│   │   query=[*].samples[*].age  pooling=query  objective=mae  weight=1.0  n_heads=4  p_mask=0.15  n_linear=1  
│   │   jitter=0.0  n_bands=8  offset=4
│   ├── sepal_length [number] 4,087 params
│   │   record/samples/sepal_length
│   │   query=[*].samples[*].sepal_length  pooling=query  objective=mae  weight=1.0  n_heads=4  p_mask=0.15  n_linear=1 
│   │   jitter=0.0  n_bands=8  offset=4
│   └── sepal_width [number] 4,087 params
│       record/samples/sepal_width
│       query=[*].samples[*].sepal_width  pooling=query  objective=mae  weight=1.0  n_heads=4  p_mask=0.15  n_linear=1  
│       jitter=0.0  n_bands=8  offset=4
└── caretaker [category] 3,857 params
    record/caretaker
    query=[*].caretaker  pooling=query  max_vocab_size=10  topk=[]  weight=1.0  n_heads=4  n_linear=1  n_bands=8  
    p_unavailable=0.01

# Resetting Nodes From Tree

Use `model.reset(*predicates)` to discard learned parameters and runtime state for selected nodes while keeping their hyperparameters intact.

This may be useful while refitting models after extreme drift is observed for specific features, such that you need to remove old vocabulary or reset the standardization hyperparameters.

In practice, this is effectively just the same thing as using `delete` on a node and then `extend` it back into the model...

In the future, `json2vec` may enable targeted arguments for the `model.reset(...)`, such that you may target a parameter group (vocabulary, counters, embeddings tables, etc.)

In [9]:
model.reset(j2v.where("type") == "number")

2026-05-28 16:18:34.348 | INFO     | json2vec.architecture.mutations:_log_node_mutation:301 - reset runtime node
2026-05-28 16:18:34.348 | INFO     | json2vec.architecture.mutations:_log_node_mutation:301 - reset runtime node
2026-05-28 16:18:34.349 | INFO     | json2vec.architecture.mutations:_log_node_mutation:301 - reset runtime node
